In [ ]:
!pip install numpy
!pip install pandas
!pip install scikit-learn
!pip install torch
!pip install torchvision
!pip install xgboost
!pip install dice_ml
!pip install lime
!pip install shap

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import lime
import shap
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import partial_dependence, permutation_importance, PartialDependenceDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from xgboost import XGBClassifier

In [ ]:
models = {
    "linear": {"Logistic Regression": LogisticRegression(random_state=42, max_iter=1000)},
    "tree": {"Random Forest": RandomForestClassifier(random_state=42, n_estimators=100), 
             "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')}
}

# Loan approval system

Dataset source: https://www.kaggle.com/datasets/taweilo/loan-approval-classification-data

In [ ]:
data1 = pd.read_csv("./Datasets/LoanApproval.csv")
data1

In [ ]:
data1.columns

In [ ]:
data1.dtypes

In [ ]:
#X = data1.drop(columns=['loan_status', 'loan_percent_income', 'previous_loan_defaults_on_file'])
X = data1.drop(columns=['loan_status'])
y = data1['loan_status']

In [ ]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
feature_names = numerical_cols + categorical_cols

In [ ]:
preprocessor_linear_data1 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ]
)

X_train_processed_linear_data1 = preprocessor_linear_data1.fit_transform(X_train)
X_test_processed_linear_data1 = preprocessor_linear_data1.transform(X_test)
X_train_final_linear = pd.DataFrame(X_train_processed_linear_data1, columns=feature_names, index=X_train.index)
X_test_final_linear = pd.DataFrame(X_test_processed_linear_data1, columns=feature_names, index=X_test.index)

In [ ]:
preprocessor_tree_data1 = ColumnTransformer(
    transformers=[
        ('num', "passthrough", numerical_cols),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ]
)

X_train_processed_tree_data1 = preprocessor_tree_data1.fit_transform(X_train)
X_test_processed_tree_data1 = preprocessor_tree_data1.transform(X_test)
X_train_final_tree = pd.DataFrame(X_train_processed_tree_data1, columns=feature_names, index=X_train.index)
X_test_final_tree = pd.DataFrame(X_test_processed_tree_data1, columns=feature_names, index=X_test.index)

In [ ]:
tools_data1 = {
    "linear": (X_train_final_linear, y_train, X_test_final_linear, y_test),
    "tree": (X_train_final_tree, y_train, X_test_final_tree, y_test)
}

In [ ]:
results_data1 = []
trained_models_data1 = {}

for name in models:
    for model_name, model in models[name].items():
        X_train_model, y_train_model, X_test_model, y_test_model = tools_data1[name]
        model.fit(X_train_model, y_train_model)
        y_pred = model.predict(X_test_model)
        accuracy = accuracy_score(y_test_model, y_pred)
        f1 = f1_score(y_test_model, y_pred)
        roc_auc = roc_auc_score(y_test_model, model.predict_proba(X_test_model)[:, 1])
        results_data1.append({
            "Model": model_name,
            "Accuracy": accuracy,
            "F1 Score": f1,
            "ROC AUC": roc_auc
        })
        trained_models_data1[model_name] = model

performance_df_data1 = pd.DataFrame(results_data1).set_index("Model")
display(performance_df_data1)

## Global explanation

### Permutation importance

In [ ]:
data1.columns[12]

In [ ]:
for name in models:
    for model_name in models[name]:
        trained_model = trained_models_data1[model_name]
        _, _, X_test_model, y_test_model = tools_data1[name]
        result = permutation_importance(trained_model, X_test_model, y_test_model, n_repeats=10, random_state=42)
        for i in result.importances_mean.argsort()[::-1]:
            #if result.importances_mean[i] - 2 * result.importances_std[i] > 0:
            print(f"Feature {data1.columns[i]}: {result.importances_mean[i]:.4f} +/- {result.importances_std[i]:.4f}")
        print("\n")
        

### Global SHAP

In [ ]:
for name in models:
    for model_name in models[name]:
        model = trained_models_data1[model_name]
        X_train_model, _, X_test_model, y_test_model = tools_data1[name]
        if name == "linear":
            explainer = shap.LinearExplainer(model, X_train_model)
        else:
            explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test_model, check_additivity=False)
        
        plt.figure(figsize=(10, 6))
        plt.title(f"Global SHAP Feature Importance ({model_name})", fontsize=14)
        shap.summary_plot(shap_values, X_test_model, plot_type="bar", show=False)
        plt.show()

        plt.figure(figsize=(10, 6))
        plt.title(f"Global SHAP Summary Plot ({model_name})", fontsize=14)
        shap.summary_plot(shap_values, X_test_model, show=False)
        plt.show()

### PDP/ICE

## Local explanation

### Local explanation with LIME

### Local explanation with SHAP

### Local explanation with DiCE

# Sharing bike demand

Dataset source: https://www.kaggle.com/competitions/bike-sharing-demand/data

In [ ]:
data2 = pd.read_csv("./Datasets/BikeSharing.csv")
data2

In [ ]:
data2.columns

In [ ]:
data2.dtypes

In [ ]:
X = data2.drop(columns=['datetime', 'count'])
y = data2['count']

In [ ]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

feature_names = numerical_cols + categorical_cols

In [ ]:
preprocessor_data2 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ]
)

X_train_processed = preprocessor_data2.fit_transform(X_train)
X_test_processed = preprocessor_data2.transform(X_test)
X_train_final = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_final = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

In [ ]:
tools_data2 = {
    "linear": (X_train_final, y_train, X_test_final, y_test),
    "tree": (X_train, y_train, X_test, y_test)
}

In [ ]:
results_data2 = []
trained_models_data2 = {}

for name in models:
    for model_name, model in models[name]:
        X_train_model, y_train_model, X_test_model, y_test_model = tools_data2[name]
        model.fit(X_train_model, y_train_model)
        y_pred = model.predict(X_test_model)
        accuracy = accuracy_score(y_test_model, y_pred)
        f1 = f1_score(y_test_model, y_pred)
        roc_auc = roc_auc_score(y_test_model, model.predict_proba(X_test_model)[:, 1])
        results_data2.append({
            "Model": model_name,
            "Accuracy": accuracy,
            "F1 Score": f1,
            "ROC AUC": roc_auc
        })
        trained_models_data2[model_name] = model

performance_df_data2 = pd.DataFrame(results_data2).set_index("Model")
display(performance_df_data2)

# Breast cancer diagnosis

In [ ]:
data3 = pd.read_csv("./Datasets/BreastCancer.csv")
data3

In [ ]:
data3.columns

In [ ]:
data3.dtypes

In [ ]:
X = data3.drop(columns=['loan_status'])
y = data3['loan_status']

In [ ]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

feature_names = numerical_cols + categorical_cols

In [ ]:
preprocessor_data3 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ]
)

X_train_processed = preprocessor_data3.fit_transform(X_train)
X_test_processed = preprocessor_data3.transform(X_test)
X_train_final = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_final = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

In [ ]:
tools_data3 = {
    "linear": (X_train_final, y_train, X_test_final, y_test),
    "tree": (X_train, y_train, X_test, y_test)
}

In [ ]:
results_data3 = []
trained_models_data3 = {}

for name in models:
    for model_name, model in models[name]:
        X_train_model, y_train_model, X_test_model, y_test_model = tools_data3[name]
        model.fit(X_train_model, y_train_model)
        y_pred = model.predict(X_test_model)
        accuracy = accuracy_score(y_test_model, y_pred)
        f1 = f1_score(y_test_model, y_pred)
        roc_auc = roc_auc_score(y_test_model, model.predict_proba(X_test_model)[:, 1])
        results_data3.append({
            "Model": model_name,
            "Accuracy": accuracy,
            "F1 Score": f1,
            "ROC AUC": roc_auc
        })
        trained_models_data3[model_name] = model

performance_df_data3 = pd.DataFrame(results_data3).set_index("Model")
display(performance_df_data3)